# Dispatch model — testing notebook

Validation scratchpad for the dispatch layer under the **nominal-LDR (A)** formulation
(robust box removed; feasibility enforced ex-post via the clip). Sections:

1. Imports & config
2. Load data / frozen forecaster / sampler
3. Build one day's dispatch inputs
4. Conditioning probe (second-moment spectrum)
5. Solver timing (plain + differentiable forward/backward)
6. **Added tests** — resolving the "2x gradient" and validating the DFL gradient

> Note: `BOX_LEVELS` / `boxes_from_quantiles` / `h_plus`/`h_minus` are **defunct** under (A)
> (no robust box, no h-selection). They are omitted below; the layer no longer takes them.


## 1. Imports & config

In [1]:
from __future__ import annotations
import sys, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import cvxpy as cp
from pyprojroot import here

ROOT_DIR        = here()
FORECASTING_DIR = ROOT_DIR / "4_forecasting"
DATA_DIR        = ROOT_DIR / "1_data" / "processed"
COPULA_DIR      = ROOT_DIR / "5_scenario_gen"
MODEL_DIR       = ROOT_DIR / "6_models"
for p in (FORECASTING_DIR, COPULA_DIR, MODEL_DIR):
    sys.path.insert(0, str(p))

from dispatch_layer   import default_fixed_params, build_problem, solve_plain, make_layer
from dispatch_wrapper import get_prices, realised_breakdown, realised_cost
#cholesky_of_second_moment
from forecasting import (reindex_and_impute, build_features, make_windows,
                         normalise_hist, denormalise_y, Baseline_Forecaster)
from copula_lib import FrozenCopulaSampler

# ---- config ----
CORNERS   = [("single", 0.0), ("single", 1.0), ("dual", 0.0), ("dual", 1.0)]
N_SCEN    = 64
DT        = 1.0
GAMMA     = 1e-4          # locked: conditions k=1, pins economic decisions to ~1e-6 (see G7/G8)
TRAIN_SOLVER = "ECOS"     # differentiable layer forward (diffcp): interior-point, accurate grads
PLAIN_SOLVER = cp.CLARABEL

# split boundaries (UTC); 2018 delivery blocks fully inside [TRAIN_START, VAL_START - 1h]
TRAIN_START = pd.Timestamp("2018-01-01 00:00:00+00:00")
VAL_START   = pd.Timestamp("2019-01-01 00:00:00+00:00")
ISSUE_HOUR  = 9
HORIZON     = 24
N_HIST      = 168                 # <-- set to your forecaster's lookback

HIST_COLS  = ["prosumption", "solar_irrad", "panel_temp", "ambient_temp"]
FEAT_COLS  = ["solar_irrad", "panel_temp", "ambient_temp"]
EXO_COLS   = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "is_weekend"]
PRICE_COLS = ["da", "imb", "up_reg_cost", "down_reg_cost"]


## 2. Load data, frozen forecaster, sampler

In [2]:
base  = pd.read_csv(DATA_DIR / "df_full.csv", parse_dates=["datetime"]).set_index("datetime")
base  = reindex_and_impute(base, HIST_COLS, freq="1h", warn_gap=6)
frame = build_features(base, feature_cols=FEAT_COLS)

windows = make_windows(
    frame, y_range=(TRAIN_START, VAL_START - pd.Timedelta(hours=1)),
    gate_aligned_only=True, issue_hour=ISSUE_HOUR,
    hist_cols=HIST_COLS, exo_cols=EXO_COLS, target_col="prosumption",
    price_cols=PRICE_COLS, n_hist=N_HIST, horizon=HORIZON,
)

SEED = 20240801
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| seed:", SEED, "| days:", len(windows.delivery_start))

# frozen baseline forecaster
ckpt  = torch.load(FORECASTING_DIR / "baseline_forecaster_best.pt", weights_only=False, map_location="cpu")
model = Baseline_Forecaster(**ckpt["model_config"])
model.load_state_dict(ckpt["state_dict"]); model.to(DEVICE); model.eval()
sc              = ckpt["scaler_stats"]
QUANTILE_LEVELS = ckpt["quantile_levels"]

# frozen copula sampler
cop     = pickle.load(open(COPULA_DIR / "frozen_copula.pkl", "rb"))
sampler = FrozenCopulaSampler(cop["Z_corr"], cop["quantile_levels"]).to(DEVICE)
assert getattr(sampler, "S", N_SCEN) == N_SCEN, "sampler scenario count must equal N_SCEN"


[impute] 66 missing values across ['prosumption', 'solar_irrad', 'panel_temp', 'ambient_temp'] after reindex
[impute]   prosumption: 20 missing, longest run 11h   <-- LONG GAP (review)
[impute]   panel_temp: 46 missing, longest run 19h   <-- LONG GAP (review)
device: cpu | seed: 20240801 | days: 365


In [3]:
def make_forecast_fn(model, scaler_stats, device):
    model.eval()
    def forecast_fn(x_hist_day, x_fut_day):
        with torch.no_grad():
            xh = normalise_hist(np.asarray(x_hist_day), scaler_stats)
            xh = torch.as_tensor(xh, dtype=torch.float32, device=device).unsqueeze(0)
            xf = torch.as_tensor(np.asarray(x_fut_day), dtype=torch.float32, device=device).unsqueeze(0)
            q_norm = model(xh, xf)                       # (1, K, Q) normalised
            q_phys = denormalise_y(q_norm, scaler_stats) # -> physical MW
        return q_phys.squeeze(0).to(torch.float64)       # (K, Q)
    return forecast_fn

forecast_fn = make_forecast_fn(model, sc, DEVICE)


## 3. Build one day's dispatch inputs

`make_day_inputs(d, price_model)` returns the layer/engine inputs for day `d`, with the
**correct** price object for the corner (single -> pi_imb; dual -> lam_up/lam_dn). Under (A)
there is no box, so no `h_plus`/`h_minus`. `solve_plain` and the layer forward select the
subset of params each corner declares, so a superset `vals` is fine.

In [4]:
def make_day_inputs(d, price_model):
    """Returns (vals, realised, prices) for one 2018 day. One forecaster+sampler pass."""
    realised  = np.asarray(windows.y[d], float)                 # (T,)
    price_day = np.asarray(windows.price[d], float)             # (T, 4)
    quantiles = forecast_fn(windows.x_hist[d], windows.x_fut[d])
    mean, xi  = sampler.mean_and_errors(quantiles)             # mean (T,), xi (N,T)
    mean_np, xi_np = mean.detach().cpu().numpy(), xi.detach().cpu().numpy()
    #Sigma  = cholesky_of_second_moment(xi_np)                  # (T,T), used at k=1
    prices = get_prices(price_day, price_model)                # {pi_da,pi_imb} | {pi_da,lam_up,lam_dn}
    vals = {"pl_hat": mean_np, "xi_samples": xi_np, **prices}
    return vals, realised, prices

D = 0                                   # the day used throughout the tests
vals, realised, prices = make_day_inputs(D, "dual")
print("day:", windows.delivery_start[D], "| vals keys:", list(vals))


day: 2018-01-01 00:00:00+00:00 | vals keys: ['pl_hat', 'xi_samples', 'pi_da', 'lam_up', 'lam_dn']


## 5. Solver timing (plain + differentiable forward/backward)

k=1 (dual) is the heavy corner. Under (A) + gamma=1e-4, ECOS/SCS are fast; Clarabel is
pathologically slow on this structure (kept for reference). **Note the `grad = None` reset
each iteration** — without it, PyTorch *accumulates* grads across solvers (this is what
produced the spurious "2x" between ECOS and SCS).

In [ ]:
fp = default_fixed_params(1.0, num_scenarios=N_SCEN, gamma=GAMMA)
b  = build_problem(fp, "dual")
keys = [p.name() for p in b.params]
lay = make_layer(b)

for solver in ["ECOS", "SCS", "CLARABEL"]:
    # plain
    t = time.perf_counter(); solve_plain(b, vals, solver=getattr(cp, solver))
    t_plain = time.perf_counter() - t
    # differentiable forward + backward (fresh args, grad NOT accumulated)
    args = [torch.tensor(np.asarray(vals[k], float), requires_grad=(k == "pl_hat")) for k in keys]
    t = time.perf_counter()
    dec = lay(*args, solver_args={"solve_method": solver})
    t_fwd = time.perf_counter() - t
    loss = realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                         realised=realised, pl_hat=args[keys.index("pl_hat")],
                         price_model="dual", clip_recourse=True, **prices)
    t = time.perf_counter(); loss.backward(); t_bwd = time.perf_counter() - t
    print(f"{solver:9s} plain={t_plain:6.2f}s  fwd={t_fwd:6.2f}s  bwd={t_bwd:6.2f}s  "
          f"total(fwd+bwd)={t_fwd+t_bwd:6.2f}s")


ECOS      plain=  6.16s  fwd=  5.60s  bwd=  8.06s  total(fwd+bwd)= 13.66s
SCS       plain=  0.75s  fwd=  5.06s  bwd=  6.34s  total(fwd+bwd)= 11.40s


## 6. Added tests

Three checks that resolve the two apparent problems from the solver saga:
the "2x gradient" (was grad **accumulation**, not a solver bug) and the finite-difference
`-188` (was `eps` below the re-solve noise floor). All run on the same day `D`, dual k=1.

### Test 1 — ECOS vs SCS gradient on identical inputs (resolves the "2x")

Fresh `args` and a **cleared grad** for each solver, identical `vals`. If the gradients now
agree, the earlier 2x was pure accumulation (`grad += grad`), not solver disagreement.

In [ ]:
grads = {}
for solver in ["ECOS", "SCS"]:
    args = [torch.tensor(np.asarray(vals[k], float), requires_grad=(k == "pl_hat")) for k in keys]
    pl = args[keys.index("pl_hat")]
    pl.grad = None                                    # <-- the fix: no accumulation
    dec = lay(*args, solver_args={"solve_method": solver})
    loss = realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                         realised=realised, pl_hat=pl,
                         price_model="dual", clip_recourse=True, **prices)
    loss.backward()
    grads[solver] = pl.grad.detach().clone()

diff = (grads["ECOS"] - grads["SCS"]).abs().max().item()
print("ECOS grad[:4]:", grads["ECOS"][:4].numpy())
print("SCS  grad[:4]:", grads["SCS"][:4].numpy())
print(f"max |ECOS - SCS| = {diff:.3e}   -> {'AGREE (2x was accumulation)' if diff < 1e-2 else 'REAL DISAGREEMENT'}")
AUTOGRAD = grads["ECOS"].numpy()                       # trusted reference for Test 3


ECOS grad[:4]: [43.59382005 55.0688097  33.57685482 17.95364316]
SCS  grad[:4]: [43.59070473 55.07673003 33.58575319 17.9488607 ]
max |ECOS - SCS| = 7.092e-02   -> REAL DISAGREEMENT


### Test 2 — forward decision agreement across solvers

The five returned decisions should agree across solvers to solver tolerance. (Recourse `D`
may be slightly looser than the here-and-now decisions — that's the known objective-inert
degeneracy at k=0-dual; at k=1 the tracking term pins `D`.)

In [ ]:
outs = {s: solve_plain(b, vals, solver=getattr(cp, s)) for s in ["ECOS", "CLARABEL"]}
for v in ["p_ch_hat", "p_dis_hat", "D_ch", "D_dis", "p_da_rel"]:
    md_ = np.abs(np.asarray(outs["ECOS"][v]) - np.asarray(outs["CLARABEL"][v])).max()
    print(f"{v:10s} max|ECOS - Clarabel| = {md_:.2e}")


p_ch_hat   max|ECOS - Clarabel| = 3.61e-04
p_dis_hat  max|ECOS - Clarabel| = 3.24e-04
D_ch       max|ECOS - Clarabel| = 7.89e-10
D_dis      max|ECOS - Clarabel| = 7.12e-10
p_da_rel   max|ECOS - Clarabel| = 4.56e-05


### Test 3 — finite-difference vs autograd (central diff, eps sweep)

The `-188` earlier came from `eps=1e-4` (forward diff) sitting **below the re-solve noise
floor**: with a true gradient ~O(40), `output-base ~ 40*eps`, which for `eps=1e-4` is ~4e-3,
smaller than solver noise. Central differences with larger `eps` lift the signal above the
floor. Expect the FD to converge toward `AUTOGRAD` as `eps` grows (until truncation error at
the largest eps). `pl_hat` is perturbed in **both** channels (layer input and the `pl_hat`
argument), so this is the full total derivative.

Caveat: the clip introduces kinks; near a saturation boundary FD averages across the kink
while autograd takes a subgradient, so small local disagreement there is expected and the
subgradient is the correct training signal.

In [ ]:
pl_idx = keys.index("pl_hat")
pl0 = np.asarray(vals["pl_hat"], float)

def loss_at(pl_vec, solver="ECOS"):
    a = [torch.tensor(np.asarray(vals[k], float)) for k in keys]
    a[pl_idx] = torch.as_tensor(np.asarray(pl_vec, float))
    dec = lay(*a, solver_args={"solve_method": solver})
    return realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                         realised=realised, pl_hat=a[pl_idx],
                         price_model="dual", clip_recourse=True, **prices).item()

for eps in [1e-1, 1e-2, 1e-3]:
    g = np.zeros(24)
    for i in range(24):
        pp = pl0.copy(); pp[i] += eps
        pm = pl0.copy(); pm[i] -= eps
        g[i] = (loss_at(pp) - loss_at(pm)) / (2 * eps)
    print(f"eps={eps:.0e}  fd[:3]={np.round(g[:3],2)}  "
          f"max|fd-autograd|={np.abs(g - AUTOGRAD).max():.2f}  "
          f"median|fd-autograd|={np.median(np.abs(g - AUTOGRAD)):.2f}")
print("\nautograd[:3]:", np.round(AUTOGRAD[:3], 2))


eps=1e-01  fd[:3]=[46.2  59.71 35.75]  max|fd-autograd|=10.11  median|fd-autograd|=1.79
eps=1e-02  fd[:3]=[44.52 58.68 33.29]  max|fd-autograd|=5.11  median|fd-autograd|=0.70
eps=1e-03  fd[:3]=[43.42 56.16 32.24]  max|fd-autograd|=14.76  median|fd-autograd|=1.34

autograd[:3]: [43.59 55.07 33.58]
